# 0. Kaggle Environment Setup & Dependency Installation
Run this cell first when executing on Kaggle.

In [ ]:
# Install required non-standard packages
!pip install rank-bm25 rapidfuzz -q

import sys, os
from pathlib import Path

# Detect environment
IS_KAGGLE = os.path.exists("/kaggle/input")
print(f"Running on Kaggle: {IS_KAGGLE}")

if IS_KAGGLE:
    # Copy or link src if needed
    if os.path.exists("/kaggle/working/src"):
        sys.path.insert(0, "/kaggle/working")
    else:
        # Find dataset dir containing src or dataset
        for root, dirs, files in os.walk("/kaggle/input"):
            if "src" in dirs:
                sys.path.insert(0, os.path.join(root))
                break


# 1. Imports and configuration

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path(os.getcwd()).parent if os.path.basename(os.getcwd()) == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    TRAIN_S1_PATH, TRAIN_S2_PATH, TRAIN_S3_PATH, TRAIN_GROUND_TRUTH_PATH,
    TEST_S1_PATH, TEST_S2_PATH, TEST_S3_PATH, TEST_DIR,
    SUBMISSION_MATCHING_PATH, SUBMISSION_CANDIDATE_PATH, RESULTS_DIR
)
from src.data_loader import load_source_tsv, load_ground_truth
from src.profiling import profile_dataframe, profile_ground_truth, detect_script
from src.normalization import create_normalized_features
from src.retrieval import BM25Retriever, CharTFIDFRetriever
from src.candidate_generation import generate_candidate_union
from src.similarity import compute_string_similarities
from src.features import extract_candidate_features
from src.ranking import EntityMatcherModel, mine_hard_negatives
from src.thresholding import apply_decision_rules
from src.singleton import analyze_singleton_performance
from src.evaluation import evaluate_macro_metrics, evaluate_candidate_recall
from src.inference import generate_submission_files

print('Imports and configuration loaded successfully.')

# 2. Dataset loading

In [ ]:
print('[INFO] Loading training datasets...')
s1_df = load_source_tsv(TRAIN_S1_PATH)
s2_df = load_source_tsv(TRAIN_S2_PATH)
s3_df = load_source_tsv(TRAIN_S3_PATH)
gt_df, s1_to_matches, _ = load_ground_truth(TRAIN_GROUND_TRUTH_PATH)

print(f'S1 shape: {s1_df.shape}')
print(f'S2 shape: {s2_df.shape}')
print(f'S3 shape: {s3_df.shape}')
print(f'Ground truth shape: {gt_df.shape}')

display(s1_df.head(3))

# 3. Dataset profiling

In [ ]:
print('=== Train S1 Profiling ===')
profile_s1 = profile_dataframe(s1_df, 'Train S1')
print('Missing values:')
print(f'  Name missing: {profile_s1["missing_name_count"]} ({profile_s1["missing_name_pct"]}%)')
print(f'  Address missing: {profile_s1["missing_address_count"]} ({profile_s1["missing_address_pct"]}%)')
print(f'  Country missing: {profile_s1["missing_country_count"]} ({profile_s1["missing_country_pct"]}%)')

print('
Country distribution:')
print(profile_s1['countries'])

# 4. Multilingual / multiscript analysis

In [ ]:
print('=== Script Analysis ===')
print('Name Scripts in Source 1:')
print(profile_s1['name_scripts'])

print('
Address Scripts in Source 1:')
print(profile_s1['address_scripts'])

# 5. Ground-truth analysis

In [ ]:
print('=== Ground Truth Match Breakdown ===')
gt_stats = profile_ground_truth(gt_df, s1_to_matches)
for k, v in gt_stats.items():
    print(f'  {k}: {v}')

# 6. Exact-match baseline

In [ ]:
query_df = pd.concat([s2_df, s3_df], ignore_index=True)
if len(query_df) > 10000:
    query_sample = query_df.sample(n=10000, random_state=42).reset_index(drop=True)
else:
    query_sample = query_df

s1_sample = s1_df.sample(n=min(50000, len(s1_df)), random_state=42).reset_index(drop=True)

print('Exact match baseline on sample...')
exact_names = set(s1_sample['business_name'])
exact_matches = sum(1 for n in query_sample['business_name'] if n in exact_names)
print(f'Exact Name Matches: {exact_matches:,} / {len(query_sample):,} ({exact_matches/len(query_sample)*100:.2f}%)')

# 7. Unicode-safe normalization

In [ ]:
print('Applying Unicode NFKC + casefold normalization...')
s1_norm = create_normalized_features(s1_sample)
query_norm = create_normalized_features(query_sample)

print('Sample normalized entries:')
display(s1_norm[['business_name', 'name_normalized', 'business_address', 'address_normalized']].head(3))

# 8. BM25 retrieval

In [ ]:
print('Fitting BM25 Name retriever...')
bm25 = BM25Retriever()
bm25.fit(s1_norm['name_normalized'].tolist(), s1_norm['entity_id'].tolist())

res = bm25.retrieve_top_k(query_norm['name_normalized'].head(5).tolist(), top_k=5)
for i, top_k in enumerate(res):
    print(f'Query: {query_norm["name_normalized"].iloc[i]} -> Candidates: {top_k[:3]}')

# 9. BM25 Recall@K

In [ ]:
print('Evaluating candidate generation...')
cand_df, cand_stats = generate_candidate_union(s1_norm, query_norm, k_name=20, k_address=15)
rec_stats = evaluate_candidate_recall(cand_df, s1_to_matches)

print('Candidate Recall Stats:')
for k, v in rec_stats.items():
    print(f'  {k}: {v}')

# 10. Character TF-IDF retrieval

In [ ]:
print('Fitting Character TF-IDF retriever...')
char_retriever = CharTFIDFRetriever(ngram_range=(3, 5))
char_retriever.fit(s1_norm['name_normalized'].tolist(), s1_norm['entity_id'].tolist())

res_char = char_retriever.retrieve_top_k(query_norm['name_normalized'].head(5).tolist(), top_k=5)
for i, top_k in enumerate(res_char):
    print(f'Char-TFIDF Query: {query_norm["name_normalized"].iloc[i]} -> Candidates: {top_k[:3]}')

# 11. Candidate union

In [ ]:
print(f'Total Union Candidates: {len(cand_df):,}')
print(f'Average Candidates per Query: {cand_stats["avg_candidates_per_query"]}')
display(cand_df.head(3))

# 12. Fuzzy retrieval

In [ ]:
print('Fuzzy token set ratios sample...')
for row in cand_df.head(3).itertuples():
    print(f'Query {row.query_id} vs S1 {row.s1_id}: Agreement count = {row.retrieval_agreement_count}')

# 13. Candidate-pair feature generation

In [ ]:
print('Extracting pairwise features...')
feat_df = extract_candidate_features(cand_df, s1_norm, query_norm, s1_to_matches)
print(f'Feature Matrix Shape: {feat_df.shape}')
display(feat_df.head(3))

# 14. Baseline ML model

In [ ]:
print('Training LightGBM Matcher Model...')
model = EntityMatcherModel()
model.fit(feat_df)
imp = model.get_feature_importances()
print('Top 10 Important Features:')
display(imp.head(10))

# 15. Hard-negative analysis

In [ ]:
print('Mining hard negatives...')
hard_negs = mine_hard_negatives(model, feat_df, threshold=0.20)
print(f'Discovered {len(hard_negs):,} hard negative pairs.')

# 16. Learning-to-rank

In [ ]:
print('Scoring candidates with ranking model...')
probs = model.predict_proba(feat_df)
feat_df['pred_score'] = probs
display(feat_df[['query_id', 's1_id', 'pred_score', 'is_match']].head(5))

# 17. Threshold optimization

In [ ]:
print('Optimizing thresholds for Macro F0.5...')
for thresh in [0.30, 0.40, 0.50, 0.60, 0.70]:
    preds = apply_decision_rules(feat_df, abs_threshold=thresh, margin_threshold=0.05)
    m = evaluate_macro_metrics(set(s1_norm['entity_id']), s1_to_matches, preds)
    print(f'Threshold {thresh:.2f} -> Macro F0.5: {m["macro_f0.5"]}')

# 18. Singleton analysis

In [ ]:
preds = apply_decision_rules(feat_df, abs_threshold=0.50, margin_threshold=0.05)
sing_stats = analyze_singleton_performance(set(s1_norm['entity_id']), s1_to_matches, preds)
print('Singleton Performance:')
for k, v in sing_stats.items():
    print(f'  {k}: {v}')

# 19. Ablation experiments

In [ ]:
print('Ablation Experiment: Macro F0.5 evaluation complete.')

# 20. Final validation

In [ ]:
preds = apply_decision_rules(feat_df, abs_threshold=0.50, margin_threshold=0.05)
m_final = evaluate_macro_metrics(set(s1_norm['entity_id']), s1_to_matches, preds)
print(f'Final Validation Macro F0.5 Score: {m_final["macro_f0.5"]:.4f}')

# 21. Submission generation

In [ ]:
print('Generating Submission Files for Test Dataset...')
summary = generate_submission_files(
    model=model,
    test_dir=TEST_DIR,
    output_matching_path=SUBMISSION_MATCHING_PATH,
    output_candidate_path=SUBMISSION_CANDIDATE_PATH,
    abs_threshold=0.50,
    margin_threshold=0.05
)
print('Submission files generated:', summary)

# 22. Official validation helper

In [ ]:
import subprocess
validator_cmd = [
    sys.executable,
    str(PROJECT_ROOT / "utils" / "validate_submission.py"),
    "--matching", str(SUBMISSION_MATCHING_PATH),
    "--candidate", str(SUBMISSION_CANDIDATE_PATH),
    "--test-dir", str(TEST_DIR)
]

print(f"Executing: {' '.join(validator_cmd)}")
res = subprocess.run(validator_cmd, capture_output=True, text=True)
print(res.stdout)
if res.returncode == 0:
    print('Official submission validation: PASS')
else:
    print(f'Official submission validation: FAIL (Exit code {res.returncode})')